# Online Shoppers Purchasing Intention: что предсказывает покупку

**Датасет:** [Online Shoppers Purchasing Intention](https://www.kaggle.com/datasets/imakash3011/online-shoppers-purchasing-intention-dataset) — 12 330 сессий интернет-магазина, 17 признаков, цель `Revenue` (закончилась ли сессия покупкой).

**Задача:** бинарная классификация с дисбалансом классов.

**Метрики:** главная — PR-AUC (average precision) и F1 по классу «покупка»; ROC-AUC, precision, recall — дополнительно; accuracy — только для сравнения (см. H1).

**План:** гипотезы → EDA → подготовка данных → базовые модели → эксперименты по гипотезам → подбор гиперпараметров → финальная оценка на test → выводы.

## Гипотезы (сформулированы ДО экспериментов)

Каждая гипотеза — конкретное утверждение с числовым критерием, который можно проверить. Цифры в критериях — мои ожидания; если не подтвердятся, это нормальный результат.

**H1 (метрики и дисбаланс).** Accuracy на этих данных вводит в заблуждение. Модель «никто ничего не покупает» (Dummy) получит accuracy ≈ 0.84–0.85 и F1 = 0 по классу «покупка». Лучшая обученная модель обгонит Dummy по accuracy не более чем на 8 п.п., но по F1 — более чем на 0.5.
*Проверка:* таблица метрик Dummy vs модели (CV и test).

**H2 (самый важный признак).** `PageValues` — самый важный признак: он на 1-м месте по permutation importance лучшей модели, а модель, обученная только на нём, даёт CV ROC-AUC ≥ 0.85.
*Проверка:* ROC-AUC на одном признаке для каждого числового признака + permutation importance на test.

**H3 (нелинейность).** Градиентный бустинг или Random Forest обгонят логистическую регрессию по PR-AUC минимум на 0.03, потому что связь признаков с покупкой нелинейна (насыщение, взаимодействия), а логрегрессия строит линейную границу.
*Проверка:* PR-AUC подобранных моделей на test (и в CV).

**H4 (мультиколлинеарность).** Пары признаков с |r| > 0.85 избыточны (ожидаются `BounceRates`–`ExitRates` и `ProductRelated`–`ProductRelated_Duration`). Удаление одного признака из каждой пары не ухудшит ROC-AUC логистической регрессии (изменение < 0.005) и не изменит качество дерева.
*Проверка:* CV ROC-AUC с этими признаками и без них + коэффициенты логрегрессии для коррелированной пары.

**H5 (переобучение дерева).** Лучшая `max_depth` по CV ROC-AUC лежит в диапазоне 3–8. При `max_depth` ≥ 15 CV ROC-AUC на 0.03 и более ниже пика, а train ROC-AUC выше 0.98.
*Проверка:* validation curve по глубине дерева.

**H6 (отбор признаков).** `SelectKBest` с k = 8 (из ~70 признаков после one-hot) не ухудшит ROC-AUC более чем на 0.01 по сравнению с полным набором — сигнал сосредоточен в малом числе признаков. Отбор делается внутри Pipeline, чтобы не было утечки.
*Проверка:* CV ROC-AUC в зависимости от k для логрегрессии и градиентного бустинга.

**H7 (порог классификации).** Порог, подобранный по максимуму F1 на out-of-fold предсказаниях train (вместо 0.5), повысит F1 по классу «покупка» на test минимум на 0.02 для лучшей модели.
*Проверка:* F1 на test при пороге 0.5 и при подобранном пороге.


## 1. Импорты и настройки

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import loguniform, randint
from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay,
                             accuracy_score, average_precision_score, f1_score, make_scorer,
                             precision_recall_curve, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import (RandomizedSearchCV, StratifiedKFold, cross_val_predict,
                                     cross_val_score, cross_validate, train_test_split, validation_curve)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', lambda v: f'{v:.3f}')

## 2. Данные и EDA

In [ ]:
DATA_PATH = 'online_shoppers_intention.csv'   # путь к скачанному CSV
df = pd.read_csv(DATA_PATH)
print('Размер:', df.shape)
display(df.head())
df.info()

In [ ]:
na = df.isna().sum()
print('Пропуски (только столбцы, где они есть):')
print(na[na > 0] if na.any() else 'пропусков нет')
print('Полных дубликатов строк:', df.duplicated().sum())
display(df.describe().T)

In [ ]:
TARGET = 'Revenue'
print(df[TARGET].value_counts())
print(df[TARGET].value_counts(normalize=True).round(3))
df[TARGET].value_counts().plot(kind='bar', title='Баланс классов (Revenue)')
plt.show()

In [ ]:
# Группы признаков (используем дальше везде)
count_cols = ['Administrative', 'Informational', 'ProductRelated']                              # число просмотренных страниц
dur_cols   = ['Administrative_Duration', 'Informational_Duration', 'ProductRelated_Duration']   # время на них
rate_cols  = ['BounceRates', 'ExitRates', 'PageValues', 'SpecialDay']
num_cols   = count_cols + dur_cols + rate_cols
cat_cols   = ['Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType']   # коды — это категории, не числа
bin_cols   = ['Weekend']

zeros = (df[num_cols] == 0).mean().rename('доля нулей')
display(pd.concat([zeros, df[num_cols].skew().rename('асимметрия')], axis=1))

fig, axes = plt.subplots(4, 3, figsize=(15, 12))
for ax, col in zip(axes.ravel(), num_cols):
    ax.hist(np.log1p(df[col]), bins=40)
    ax.set_title(f'log1p({col})')
for ax in axes.ravel()[len(num_cols):]:
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
display(df.groupby(TARGET)[num_cols].mean().T)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['PageValues', 'ExitRates', 'ProductRelated_Duration']):
    sns.boxplot(x=df[TARGET], y=np.log1p(df[col]), ax=ax)
    ax.set_title(f'log1p({col}) по Revenue')
plt.tight_layout(); plt.show()

In [ ]:
corr = df[num_cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Корреляции числовых признаков'); plt.show()

print('Пары с |r| > 0.8:')
for i, a in enumerate(num_cols):
    for b in num_cols[i + 1:]:
        if abs(corr.loc[a, b]) > 0.8:
            print(f'  {a} — {b}: {corr.loc[a, b]:.2f}')

In [ ]:
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'June', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
base_rate = df[TARGET].mean()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, col in zip(axes.ravel(), ['Month', 'VisitorType', 'Weekend', 'TrafficType']):
    stats = df.groupby(col)[TARGET].agg(['mean', 'count'])
    if col == 'Month':
        stats = stats.reindex([m for m in month_order if m in stats.index])
    stats['mean'].plot(kind='bar', ax=ax, title=f'Доля покупок по {col}')
    ax.axhline(base_rate, color='red', ls='--', label='средняя по датасету')
    ax.set_ylabel('P(Revenue=1)')
axes[0, 0].legend()
plt.tight_layout(); plt.show()
display(df.groupby('VisitorType')[TARGET].agg(['mean', 'count']))

**Выводы по EDA** *(заполни своими словами после просмотра графиков)*

- Баланс классов: ...
- Распределения: ... (скошенность, доля нулей — зачем `log1p`)
- Что отличает покупающие сессии от остальных: ...
- Сильно коррелирующие признаки: ...
- Категории (месяц, тип посетителя, ...): ...
- Пропуски и дубликаты: ... (что делаем и почему)

## 3. Подготовка данных

Решения и причины:
- **Дубликаты удаляем до разбиения**: одинаковые строки могли бы попасть и в train, и в test — это утечка.
- **Коды ОС, браузера, региона, типа трафика — категории**, а не числа (порядок кодов ничего не значит), поэтому one-hot.
- **Числа с тяжёлым хвостом и множеством нулей** (счётчики страниц, длительности, `PageValues`): для линейной модели `log1p` + стандартизация; деревьям масштаб не важен.
- **Масштабирование и кодирование — внутри `Pipeline`**, поэтому на каждой CV-итерации параметры считаются только по train-части (без утечки).
- **Test откладываем сразу и не трогаем до финальной оценки.** Все подборы (гиперпараметры, порог, k для отбора признаков) — только на train через кросс-валидацию.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
df['Weekend'] = df['Weekend'].astype(int)
df[cat_cols] = df[cat_cols].astype(str)

X = df.drop(columns=TARGET)
y = df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print('train:', X_train.shape, 'test:', X_test.shape)
print('доля покупок train / test:', round(y_train.mean(), 3), '/', round(y_test.mean(), 3))

In [ ]:
skewed_cols = count_cols + dur_cols + ['PageValues']   # тяжёлый хвост + много нулей -> log1p
other_num_cols = ['BounceRates', 'ExitRates', 'SpecialDay']

def make_preprocessor(linear=True, drop=()):
    """linear=True: log1p + стандартизация (нужно логрегрессии); False: числа как есть (деревьям масштаб не важен).
    drop — список числовых признаков, которые нужно убрать (для эксперимента H4)."""
    skew = [c for c in skewed_cols if c not in drop]
    other = [c for c in other_num_cols if c not in drop]
    if linear:
        skew_tf = Pipeline([('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
                            ('scale', StandardScaler())])
        other_tf = StandardScaler()
    else:
        skew_tf, other_tf = 'passthrough', 'passthrough'
    return ColumnTransformer([
        ('skew', skew_tf, skew),
        ('num', other_tf, other),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('bin', 'passthrough', bin_cols),
    ], sparse_threshold=0)

# Кросс-валидация и метрики: стратифицированная, чтобы доля покупок была одинаковой в каждом фолде
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    'accuracy': 'accuracy',
    'f1': make_scorer(f1_score, zero_division=0),
    'precision': make_scorer(precision_score, zero_division=0),
    'recall': make_scorer(recall_score, zero_division=0),
    'roc_auc': 'roc_auc',
    'pr_auc': 'average_precision',
}

def cv_row(name, model, X, y):
    res = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    row = {'model': name}
    row.update({m: res[f'test_{m}'].mean() for m in scoring})
    row['roc_auc_std'] = res['test_roc_auc'].std()
    row['pr_auc_std'] = res['test_pr_auc'].std()
    return row

## 4. Базовые модели (H1, первый взгляд на H3)

Начинаем с **baseline** — глупой модели, с которой сравниваем всё остальное, — затем берём три разных семейства моделей с параметрами по умолчанию: линейную (LogReg), одно дерево и ансамбли (Random Forest, градиентный бустинг). Всё оцениваем кросс-валидацией на train.

In [ ]:
models = {
    'Dummy': Pipeline([('prep', make_preprocessor(False)),
                       ('clf', DummyClassifier(strategy='most_frequent'))]),
    'LogReg': Pipeline([('prep', make_preprocessor(True)),
                        ('clf', LogisticRegression(max_iter=2000))]),
    'DecisionTree': Pipeline([('prep', make_preprocessor(False)),
                              ('clf', DecisionTreeClassifier(random_state=RANDOM_STATE))]),
    'RandomForest': Pipeline([('prep', make_preprocessor(False)),
                              ('clf', RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE))]),
    'HistGB': Pipeline([('prep', make_preprocessor(False)),
                        ('clf', HistGradientBoostingClassifier(random_state=RANDOM_STATE))]),
}

cv_default = pd.DataFrame([cv_row(n, m, X_train, y_train) for n, m in models.items()]).set_index('model')
display(cv_default)

In [ ]:
# Проверка H1: accuracy vs F1
dummy = cv_default.loc['Dummy']
best = cv_default.drop('Dummy')['pr_auc'].idxmax()
b = cv_default.loc[best]
print(f"Accuracy: Dummy {dummy['accuracy']:.3f} vs {best} {b['accuracy']:.3f}  -> прирост {(b['accuracy'] - dummy['accuracy']) * 100:.1f} п.п.")
print(f"F1:       Dummy {dummy['f1']:.3f} vs {best} {b['f1']:.3f}  -> прирост {b['f1'] - dummy['f1']:.3f}")

## 5. H2 — какие признаки важнее всего (часть 1: один признак за раз)

Для каждого числового признака обучаем бустинг **только на нём** и смотрим CV ROC-AUC. Это показывает, сколько информации несёт каждый признак сам по себе (0.5 = ничего).

In [ ]:
single = {}
for col in num_cols:
    single[col] = cross_val_score(HistGradientBoostingClassifier(random_state=RANDOM_STATE),
                                  X_train[[col]], y_train, cv=cv, scoring='roc_auc', n_jobs=-1).mean()
single = pd.Series(single).sort_values(ascending=False)
display(single.to_frame('ROC-AUC (только этот признак)'))
single.sort_values().plot(kind='barh', figsize=(7, 4), title='ROC-AUC модели на одном признаке')
plt.axvline(0.5, color='red', ls='--'); plt.show()

## 6. H5 — глубина дерева и переобучение

Кривая валидации: как меняется качество на train и на CV при росте `max_depth`. Расхождение кривых = переобучение.

In [ ]:
depths = list(range(1, 26))
tree_pipe = Pipeline([('prep', make_preprocessor(False)),
                      ('clf', DecisionTreeClassifier(random_state=RANDOM_STATE))])
tr, va = validation_curve(tree_pipe, X_train, y_train, param_name='clf__max_depth',
                          param_range=depths, cv=cv, scoring='roc_auc', n_jobs=-1)
tr_m, va_m, va_s = tr.mean(axis=1), va.mean(axis=1), va.std(axis=1)

plt.figure(figsize=(8, 5))
plt.plot(depths, tr_m, 'o-', label='train')
plt.plot(depths, va_m, 'o-', label='CV')
plt.fill_between(depths, va_m - va_s, va_m + va_s, alpha=0.2)
plt.xlabel('max_depth'); plt.ylabel('ROC-AUC'); plt.legend(); plt.title('Переобучение дерева')
plt.show()

best_depth = depths[int(va_m.argmax())]
print(f'Лучшая глубина по CV: {best_depth} (CV ROC-AUC = {va_m.max():.3f})')
for d in (15, 25):
    print(f'depth={d}: train {tr_m[d - 1]:.3f}, CV {va_m[d - 1]:.3f}, падение от пика {va_m.max() - va_m[d - 1]:.3f}')

## 7. H4 — мультиколлинеарность

Находим пары признаков с |r| > 0.85 (по train), из каждой пары убираем тот признак, который слабее связан с таргетом, и сравниваем качество с полным набором.

In [ ]:
CORR_THR = 0.85
corr_tr = X_train[num_cols].corr().abs()
target_corr = X_train[num_cols].apply(lambda s: abs(np.corrcoef(s, y_train)[0, 1]))
pairs = [(a, b) for i, a in enumerate(num_cols) for b in num_cols[i + 1:] if corr_tr.loc[a, b] > CORR_THR]
to_drop = sorted({a if target_corr[a] < target_corr[b] else b for a, b in pairs})
print('Пары с |r| >', CORR_THR, ':')
for a, b in pairs:
    print(f'  {a} — {b}: r = {corr_tr.loc[a, b]:.2f}')
print('Удаляем (слабее связан с таргетом):', to_drop)

rows = []
for label, drop in [('все признаки', ()), ('без ' + ', '.join(to_drop), tuple(to_drop))]:
    for name, clf, linear in [('LogReg', LogisticRegression(max_iter=2000), True),
                              ('DecisionTree', DecisionTreeClassifier(max_depth=best_depth, random_state=RANDOM_STATE), False)]:
        pipe = Pipeline([('prep', make_preprocessor(linear, drop)), ('clf', clf)])
        r = cv_row(name, pipe, X_train, y_train)
        rows.append({'модель': name, 'набор': label, 'roc_auc': r['roc_auc'], 'roc_auc_std': r['roc_auc_std'], 'pr_auc': r['pr_auc']})
display(pd.DataFrame(rows))

In [ ]:
# Коэффициенты логрегрессии для скоррелированных признаков (на полном наборе)
if pairs:
    pipe_full = Pipeline([('prep', make_preprocessor(True)),
                          ('clf', LogisticRegression(max_iter=2000))]).fit(X_train, y_train)
    names = pipe_full['prep'].get_feature_names_out()
    coefs = pd.Series(pipe_full['clf'].coef_[0], index=names)
    involved = sorted({c for p in pairs for c in p})
    display(coefs[[n for n in names if n.split('__', 1)[1] in involved]].to_frame('коэффициент'))

## 8. Подбор гиперпараметров

`RandomizedSearchCV` по 5-fold CV на train, метрика — **PR-AUC** (average precision): она не зависит от порога и подходит для несбалансированных классов. Test здесь не используется.

Замечание: лучший CV-скор после перебора слегка оптимистичен (мы выбрали максимум из шумных оценок) — честная оценка будет на test.

In [ ]:
N_ITER = 20

base_pipes = {
    'LogReg': Pipeline([('prep', make_preprocessor(True)), ('clf', LogisticRegression(max_iter=3000))]),
    'DecisionTree': Pipeline([('prep', make_preprocessor(False)), ('clf', DecisionTreeClassifier(random_state=RANDOM_STATE))]),
    'RandomForest': Pipeline([('prep', make_preprocessor(False)), ('clf', RandomForestClassifier(random_state=RANDOM_STATE))]),
    'HistGB': Pipeline([('prep', make_preprocessor(False)), ('clf', HistGradientBoostingClassifier(random_state=RANDOM_STATE))]),
}
param_dists = {
    'LogReg': {'clf__C': loguniform(1e-3, 1e2), 'clf__class_weight': [None, 'balanced']},
    'DecisionTree': {'clf__max_depth': randint(2, 15), 'clf__min_samples_leaf': randint(1, 50),
                     'clf__criterion': ['gini', 'entropy']},
    'RandomForest': {'clf__n_estimators': randint(100, 400), 'clf__max_depth': [None, 5, 10, 15],
                     'clf__min_samples_leaf': randint(1, 20), 'clf__max_features': ['sqrt', 0.3]},
    'HistGB': {'clf__learning_rate': loguniform(0.01, 0.3), 'clf__max_depth': [None, 3, 5, 8],
               'clf__max_leaf_nodes': randint(8, 64), 'clf__l2_regularization': loguniform(1e-3, 10),
               'clf__min_samples_leaf': randint(10, 100)},
}

searches = {}
for name, pipe in base_pipes.items():
    s = RandomizedSearchCV(pipe, param_dists[name], n_iter=N_ITER, scoring='average_precision',
                           cv=cv, n_jobs=-1, random_state=RANDOM_STATE)
    s.fit(X_train, y_train)
    searches[name] = s
    print(f'{name:13s} CV PR-AUC = {s.best_score_:.3f}   {s.best_params_}')

best_name = max(searches, key=lambda n: searches[n].best_score_)
print('\nЛучшая модель по CV PR-AUC:', best_name)

## 9. H6 — отбор признаков (`SelectKBest`)

Отбор стоит **внутри Pipeline** — значит, на каждой CV-итерации признаки выбираются только по train-части фолда. Если отобрать признаки по всему train до кросс-валидации, информация о валидационных фолдах «просочится» в отбор.

In [ ]:
ks = [3, 5, 8, 12, 20, 'all']
rows = []
for name, clf in [('LogReg', LogisticRegression(max_iter=2000)),
                  ('HistGB', HistGradientBoostingClassifier(random_state=RANDOM_STATE))]:
    for k in ks:
        steps = [('prep', make_preprocessor(True))]
        if k != 'all':
            steps.append(('select', SelectKBest(f_classif, k=k)))
        steps.append(('clf', clone(clf)))
        r = cv_row(name, Pipeline(steps), X_train, y_train)
        rows.append({'model': name, 'k': str(k), 'roc_auc': r['roc_auc'], 'pr_auc': r['pr_auc']})
sel = pd.DataFrame(rows)
order = [str(k) for k in ks]
tab = sel.pivot(index='k', columns='model', values=['roc_auc', 'pr_auc']).loc[order]
display(tab)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ['roc_auc', 'pr_auc']):
    for name in ['LogReg', 'HistGB']:
        sub = sel[sel.model == name].set_index('k').loc[order]
        ax.plot(order, sub[metric], 'o-', label=name)
    ax.set_xlabel('k (число признаков после one-hot)'); ax.set_ylabel(metric); ax.legend()
plt.tight_layout(); plt.show()

# Какие признаки выбрал SelectKBest при k=8 (обучено только на train)
prep = make_preprocessor(True).fit(X_train, y_train)
selector = SelectKBest(f_classif, k=8).fit(prep.transform(X_train), y_train)
print('k=8:', list(np.array(prep.get_feature_names_out())[selector.get_support()]))

## 10. H7 — подбор порога классификации (только train)

Для каждой подобранной модели получаем **out-of-fold** вероятности на train (каждый объект предсказан моделью, которая его не видела) и выбираем порог, максимизирующий F1 по классу «покупка». Test по-прежнему не трогаем.

In [ ]:
thr_by_model = {}
fig, ax = plt.subplots(figsize=(8, 5))
for name, s in searches.items():
    oof = cross_val_predict(clone(s.best_estimator_), X_train, y_train, cv=cv,
                            method='predict_proba', n_jobs=-1)[:, 1]
    prec, rec, thr = precision_recall_curve(y_train, oof)
    f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
    thr_by_model[name] = float(thr[f1s.argmax()])
    ax.plot(thr, f1s, label=f'{name} (лучший порог {thr_by_model[name]:.2f}, F1 {f1s.max():.3f})')
ax.axvline(0.5, color='gray', ls='--')
ax.set_xlabel('порог'); ax.set_ylabel('F1 (out-of-fold, train)'); ax.legend()
plt.show()

## 11. Финальная оценка на test (H1, H2, H3, H7)

Единственное место, где используется test. Модели уже выбраны и обучены на train, пороги подобраны на train.

In [ ]:
def test_row(name, model, thr=0.5):
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= thr).astype(int)
    return {'model': name, 'порог': thr,
            'accuracy': accuracy_score(y_test, pred),
            'precision': precision_score(y_test, pred, zero_division=0),
            'recall': recall_score(y_test, pred, zero_division=0),
            'f1': f1_score(y_test, pred, zero_division=0),
            'roc_auc': roc_auc_score(y_test, proba),
            'pr_auc': average_precision_score(y_test, proba)}

dummy_fit = clone(models['Dummy']).fit(X_train, y_train)
rows = [test_row('Dummy', dummy_fit)]
for name, s in searches.items():
    rows.append(test_row(name, s.best_estimator_, 0.5))
    rows.append(test_row(name + ' (порог подобран)', s.best_estimator_, thr_by_model[name]))
final = pd.DataFrame(rows).set_index('model')
display(final)

In [ ]:
# Кривые ROC и Precision-Recall на test
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, s in searches.items():
    RocCurveDisplay.from_estimator(s.best_estimator_, X_test, y_test, name=name, ax=axes[0])
    PrecisionRecallDisplay.from_estimator(s.best_estimator_, X_test, y_test, name=name, ax=axes[1])
axes[0].set_title('ROC'); axes[1].set_title('Precision-Recall')
plt.show()

# Матрицы ошибок лучшей модели: порог 0.5 и подобранный
best_model = searches[best_name].best_estimator_
proba = best_model.predict_proba(X_test)[:, 1]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, thr in zip(axes, [0.5, thr_by_model[best_name]]):
    ConfusionMatrixDisplay.from_predictions(y_test, (proba >= thr).astype(int), ax=ax, colorbar=False)
    ax.set_title(f'{best_name}, порог {thr:.2f}')
plt.tight_layout(); plt.show()

In [ ]:
# H2 (часть 2): permutation importance лучшей модели на test — насколько падает PR-AUC, если перемешать признак
pi = permutation_importance(best_model, X_test, y_test, scoring='average_precision',
                            n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=X_test.columns).sort_values()
imp.plot(kind='barh', figsize=(8, 6), title=f'Permutation importance ({best_name}): падение PR-AUC')
plt.show()
display(imp.sort_values(ascending=False).head(8).to_frame('падение PR-AUC'))

## 12. Выводы по гипотезам

*Заполни после запуска всех ячеек: для каждой гипотезы — вердикт (подтвердилась / не подтвердилась / частично), числа из твоих таблиц и объяснение **почему** так получилось. Не просто «R² = 0.85», а что это значит.*

**H1 (accuracy vs дисбаланс).** Вердикт: ___. Числа: Dummy accuracy = ___, F1 = ___; лучшая модель accuracy = ___, F1 = ___. Почему: ___

**H2 (PageValues).** Вердикт: ___. Числа: ROC-AUC на одном признаке = ___; место в permutation importance = ___. Почему: ___ (что вообще означает `PageValues` и почему он так связан с покупкой?)

**H3 (бустинг/RF vs LogReg).** Вердикт: ___. Числа: PR-AUC LogReg = ___, лучшая нелинейная = ___, разница = ___. Почему: ___

**H4 (мультиколлинеарность).** Вердикт: ___. Числа: ROC-AUC LogReg с признаками / без = ___ / ___; дерево = ___ / ___; коэффициенты пары = ___. Почему: ___

**H5 (глубина дерева).** Вердикт: ___. Числа: лучшая глубина = ___; при depth=15: train = ___, CV = ___. Почему: ___

**H6 (SelectKBest).** Вердикт: ___. Числа: k=8 vs all для LogReg и HistGB = ___. Почему: ___ (какие признаки выбраны и почему этого хватает?)

**H7 (порог).** Вердикт: ___. Числа: F1 на test при 0.5 = ___, при подобранном пороге = ___. Почему: ___

### Общие выводы
- Какая модель лучшая и насколько стабильна оценка (std по фолдам, разница CV и test): ___
- Что бы я сделал дальше (например, проверка по времени: обучение на ранних месяцах, тест на поздних): ___
- Ограничения: ___ (один датасет, нет информации о пользователях, сессии могут принадлежать одному человеку, ...)